## Reading the CSV

In [46]:
import pandas as pd

df = pd.read_csv("cars.csv")
df[['brand', 'series', 'year', 'mileage', 'price']].head()

,brand,series,year,mileage,price
0,Ford,Focus,2018,180.000 km,1.049.000 TL
1,Ford,Focus,2010,357.000 km,555.750 TL
2,Citroen,C-Elysée,2014,225.000 km,595.000 TL
3,Mercedes - Benz,C,2014,75.000 km,2.059.000 TL
4,Volvo,S60,2014,193.000 km,1.150.000 TL


## Cleaning the Strings

### We want to clean the price and mileage data and convert them to floats

In [47]:
df["price"] = df["price"].str.replace(" TL", "").str.replace(".", "").astype(float)
price.head()

0    1049000.0
1     555750.0
2     595000.0
3    2059000.0
4    1150000.0
Name: price, dtype: float64

In [48]:
df["mileage"] = df["mileage"].str.replace(" km", "").str.replace(".", "").astype(float)
df["mileage"].head()

0    180000.0
1    357000.0
2    225000.0
3     75000.0
4    193000.0
Name: mileage, dtype: float64

## Concatenate Brand & Series

### I decided to concatenate these two since each brand's series are special to that brand.

In [49]:
df["brand_series"] = df["brand"] + "_" + df["series"]
df["brand_series"].head()

0           Ford_Focus
1           Ford_Focus
2     Citroen_C-Elysée
3    Mercedes - Benz_C
4            Volvo_S60
Name: brand_series, dtype: str

## Encoding Brand & Series

### We want to encode these to labels for our model to understand since working with strings does not make sense.

In [50]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["brand_series_encoded"] = le.fit_transform(df["brand_series"])
df["brand_series_encoded"].head()

0     43
1     43
2      7
3     67
4    113
Name: brand_series_encoded, dtype: int64

## Creating the Train & Test Splits

In [51]:
X = df[['brand_series_encoded', 'year', 'mileage']]
y = df["price"]

print(X.head())
print(y.head())

   brand_series_encoded  year   mileage
0                    43  2018  180000.0
1                    43  2010  357000.0
2                     7  2014  225000.0
3                    67  2014   75000.0
4                   113  2014  193000.0
0    1049000.0
1     555750.0
2     595000.0
3    2059000.0
4    1150000.0
Name: price, dtype: float64


In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

## Training the baseline decision tree

In [53]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print("Decision Tree MAE:", mean_absolute_error(y_test, y_pred))

Decision Tree MAE: 221977.789893617


In [54]:
df["price"].describe()

count    1.877000e+03
mean     1.116635e+06
std      1.353465e+06
min      5.000000e+04
25%      5.059000e+05
50%      8.100000e+05
75%      1.305000e+06
max      3.690000e+07
Name: price, dtype: float64

In [72]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=5)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print("XGBoost MAE:", mean_absolute_error(y_test, y_pred_xgb))

XGBoost MAE: 171519.14613115028
